# 06 · Predictive Modelling — Win/Loss Prediction & Strategy Backtest
> Can sentiment predict trade outcomes? Can a sentiment-gated strategy beat the baseline?

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / "src"))

import pandas as pd
import plotly.express as px
from modelling import prepare_ml_data, train_xgboost, train_lightgbm, sentiment_gated_strategy_backtest
from visualisation import plot_feature_importance

PROCESSED_DIR = pathlib.Path().resolve().parent / "data" / "processed"
merged = pd.read_csv(PROCESSED_DIR / "merged_dataset.csv")


## 1. Prepare ML Features & Target

In [ ]:
X, y = prepare_ml_data(merged)
print(f"Class balance: {y.value_counts(normalize=True).to_dict()}")
X.describe()


## 2. Train XGBoost

In [ ]:
xgb_results = train_xgboost(X, y, cv_folds=5)
print(f"XGBoost  AUC: {xgb_results['cv_auc_mean']} ± {xgb_results['cv_auc_std']}")
print(f"XGBoost  Acc: {xgb_results['cv_acc_mean']}")


## 3. Train LightGBM

In [ ]:
lgb_results = train_lightgbm(X, y, cv_folds=5)
print(f"LightGBM AUC: {lgb_results['cv_auc_mean']} ± {lgb_results['cv_auc_std']}")
print(f"LightGBM Acc: {lgb_results['cv_acc_mean']}")


## 4. Feature Importance

In [ ]:
fig = plot_feature_importance(xgb_results["feature_importance"], save=True)
fig.show()


## 5. Sentiment-Gated Strategy Backtest

In [ ]:
# Aggregate to daily PnL
daily_pnl = merged.groupby(["trade_date", "regime"])["closedpnl"].sum().reset_index()
daily_pnl.columns = ["trade_date", "regime", "daily_pnl"]
daily_pnl = daily_pnl.sort_values("trade_date")

backtest_df = sentiment_gated_strategy_backtest(daily_pnl)
backtest_df.head()


## 6. Equity Curve — Strategy vs Baseline

In [ ]:
fig = px.line(backtest_df, x="trade_date",
              y=["cum_strategy_pnl", "cum_baseline_pnl"],
              title="Equity Curve: Sentiment-Gated Strategy vs Baseline",
              labels={"value": "Cumulative PnL", "variable": "Strategy"},
              template="plotly_dark")
fig.show()


## 7. Key Insights Summary

In [ ]:
print("=" * 60)
print("  PRIME TRADE — KEY FINDINGS")
print("=" * 60)
print(f"  XGBoost AUC   : {xgb_results['cv_auc_mean']:.3f}")
print(f"  LightGBM AUC  : {lgb_results['cv_auc_mean']:.3f}")
print(f"  Top sentiment feature importance:")
print(xgb_results['feature_importance'].head(5).to_string(index=False))
print()
print(f"  Strategy vs Baseline:")
print(f"  Strategy total PnL  : {backtest_df['cum_strategy_pnl'].iloc[-1]:,.2f}")
print(f"  Baseline total PnL  : {backtest_df['cum_baseline_pnl'].iloc[-1]:,.2f}")


## ✅ Analysis Complete!
> See `outputs/figures/` for all charts and `outputs/models/` for saved models.